In [0]:
from pyspark.sql.functions import current_timestamp, col

# Define parameters via widgets
dbutils.widgets.text("catalog", "dbr_dev", "1. Catalog Name")
dbutils.widgets.text("schema", "valeriimatviiv_bronze", "2. Target Schema")
dbutils.widgets.text("volume", "market_radar_landing", "3. Landing Volume")

# Retrieve widget values
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume = dbutils.widgets.get("volume")

base_volume_path = f"/Volumes/{catalog}/{schema}/{volume}"
landing_price_path = f"{base_volume_path}/landing/nasdaq_price"
target_table = f"{catalog}.{schema}.nasdaq_price_bronze"

# 1. Drop existing table to clear stale schema conflicts
spark.sql(f"DROP TABLE IF EXISTS {target_table}")

# 2. Batch ingestion from landed CSV
df_price_batch = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(landing_price_path)
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingest_timestamp", current_timestamp())
)

# 3. Idempotent batch write with schema overwrite enabled
(
    df_price_batch.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

print(f"Successfully processed batch into Bronze table: '{target_table}'")

In [0]:
# catalog = dbutils.widgets.get("catalog")
# schema = dbutils.widgets.get("schema")
# target_table = f"{catalog}.{schema}.nasdaq_price_bronze"

# df_bronze_price = spark.read.table(target_table)

# print(f"--- Bronze Price Table Record Count: {df_bronze_price.count()} ---")
# print("--- Schema Breakdown ---")
# df_bronze_price.printSchema()

# print("--- Preview Data ---")
# display(df_bronze_price.limit(10))